Sophie Krajmalnik srk0219 sophiekrajmalnik@my.unt.edu

In [1]:
import pandas as pd
import numpy as np

In [2]:
#Read in each file
assessments = pd.read_csv('assessments.csv')
courses = pd.read_csv('courses.csv')
student_assessment = pd.read_csv('studentAssessment.csv')
student_info = pd.read_csv('studentInfo.csv')
student_registration = pd.read_csv('studentRegistration.csv')
vle = pd.read_csv('vle.csv')

In [3]:
student_vle = pd.read_csv('studentVle.csv')

In [4]:
#View structure of each file
print(assessments.head())
print("Entries: ", len(assessments))
#Link to student_assessment via id_assessment

  code_module code_presentation  id_assessment assessment_type date  weight
0         AAA             2013J           1752             TMA   19    10.0
1         AAA             2013J           1753             TMA   54    20.0
2         AAA             2013J           1754             TMA  117    20.0
3         AAA             2013J           1755             TMA  166    20.0
4         AAA             2013J           1756             TMA  215    30.0
Entries:  206


In [5]:
print(courses.head())
print("Entries: ", len(courses))
#Link to assessments  via code module/presentation

  code_module code_presentation  module_presentation_length
0         AAA             2013J                         268
1         AAA             2014J                         269
2         BBB             2013J                         268
3         BBB             2014J                         262
4         BBB             2013B                         240
Entries:  22


In [6]:
print(student_assessment.head())
print("Entries: ", len(student_assessment))
#SCORE IDENTIFIED AS BENEFICIAL FEATURE

   id_assessment  id_student  date_submitted  is_banked score
0           1752       11391              18          0    78
1           1752       28400              22          0    70
2           1752       31604              17          0    72
3           1752       32885              26          0    69
4           1752       38053              19          0    79
Entries:  173912


In [7]:
print(student_info.head())
print("Entries: ", len(student_info))
#Link to assessments, courses via code module/presentation
#Link to student_assessments via id_student

  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%     55<=                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55                     0   
4     Lower Than A Level   50-60%     0-35                     0   

   studied_credits disability final_result  
0              240          N         Pass  
1               60          N         Pass  

In [8]:
print(student_registration.head())
print("Entries: ", len(student_registration))
#Link to assessments, courses via code module/presentation
#Link to student_info, student_assessment via id_student

  code_module code_presentation  id_student date_registration  \
0         AAA             2013J       11391              -159   
1         AAA             2013J       28400               -53   
2         AAA             2013J       30268               -92   
3         AAA             2013J       31604               -52   
4         AAA             2013J       32885              -176   

  date_unregistration  
0                   ?  
1                   ?  
2                  12  
3                   ?  
4                   ?  
Entries:  32593


In [9]:
print(vle.head())
print("Entries: ", len(vle))
#Link to assessments, courses via code module/presentation

   id_site code_module code_presentation activity_type week_from week_to
0   546943         AAA             2013J      resource         ?       ?
1   546712         AAA             2013J     oucontent         ?       ?
2   546998         AAA             2013J      resource         ?       ?
3   546888         AAA             2013J           url         ?       ?
4   547035         AAA             2013J      resource         ?       ?
Entries:  6364


In [10]:
print(student_vle.head())
#Link to courses, assessments via code module/presentation
#Link to student info, student assessment via id_student
#Link to VLE via id_site
#SUM_CLICK IDENTIFIED AS BENEFICIAL FEATURE
print("Entries: ", len(student_vle))

  code_module code_presentation  id_student  id_site  date  sum_click
0         AAA             2013J       28400   546652   -10          4
1         AAA             2013J       28400   546652   -10          1
2         AAA             2013J       28400   546652   -10          1
3         AAA             2013J       28400   546614   -10         11
4         AAA             2013J       28400   546714   -10          1
Entries:  10655280


1. Preprocess the dataset (as done in the paper):


*   Merge the OULAD tables

*   Create demographic, behavioral, and assessment  features
*   Handle missing values using median imputation


*   Normalize numerical features (MinMaxScaler)
*   One-hot encode categorical features


In [11]:
#Quote from paper: "As a result, the sum-click and score features were grouped,
# and the total portal clicks for a particular student throughout the year and their average grade were calculated"
#Calculate total portal clicks - behavioral feature
merged_vle = student_vle.merge(vle, on=['id_site', 'code_module', 'code_presentation'], how='left')
print(merged_vle.head())
vle_features = merged_vle.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
#Like in SQL, group by this unique combo of ID/Module/Presentation/ActivityType, then sum the sum_click feature for each combo.
#Reset index maintains structure as dataframe
vle_features.rename(columns={'sum_click': 'total_clicks'}, inplace=True)
#Replace sum_clicks with total_clicks
print(vle_features.head())
print("Entries: ", len(vle_features))

  code_module code_presentation  id_student  id_site  date  sum_click  \
0         AAA             2013J       28400   546652   -10          4   
1         AAA             2013J       28400   546652   -10          1   
2         AAA             2013J       28400   546652   -10          1   
3         AAA             2013J       28400   546614   -10         11   
4         AAA             2013J       28400   546714   -10          1   

  activity_type week_from week_to  
0       forumng         ?       ?  
1       forumng         ?       ?  
2       forumng         ?       ?  
3      homepage         ?       ?  
4     oucontent         ?       ?  
   id_student code_module code_presentation  total_clicks
0        6516         AAA             2014J          2791
1        8462         DDD             2013J           646
2        8462         DDD             2014J            10
3       11391         AAA             2013J           934
4       23629         BBB             2013B           1

In [12]:
#Quote from paper: "As a result, the sum-click and score features were grouped,
# and the total portal clicks for a particular student throughout the year and their average grade were calculated"

final_assessment = student_assessment.merge(assessments, on=['id_assessment'], how='inner')
print(final_assessment.info())
# Convert 'score' to numeric to be able to perform mean function
final_assessment['score'] = pd.to_numeric(final_assessment['score'], errors='coerce')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id_assessment      173912 non-null  int64  
 1   id_student         173912 non-null  int64  
 2   date_submitted     173912 non-null  int64  
 3   is_banked          173912 non-null  int64  
 4   score              173912 non-null  object 
 5   code_module        173912 non-null  object 
 6   code_presentation  173912 non-null  object 
 7   assessment_type    173912 non-null  object 
 8   date               173912 non-null  object 
 9   weight             173912 non-null  float64
dtypes: float64(1), int64(4), object(5)
memory usage: 13.3+ MB
None


In [13]:
final_assessment.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19,10.0


In [14]:
# Calculate the mean score - assessment feature
avg_scores = final_assessment.groupby(['id_student', 'code_module', 'code_presentation'])['score'].mean().reset_index()
avg_scores.rename(columns={'score': 'avg_grade'}, inplace=True)
print(avg_scores.head())
print("Entries: ", len(avg_scores))

   id_student code_module code_presentation  avg_grade
0        6516         AAA             2014J  61.800000
1        8462         DDD             2013J  87.666667
2        8462         DDD             2014J  86.500000
3       11391         AAA             2013J  82.000000
4       23629         BBB             2013B  82.500000
Entries:  25843


In [15]:
#Merge datasets into one
master_df = student_info.merge(student_registration, on=['id_student', 'code_module', 'code_presentation'], how='inner')
master_df = master_df.merge(avg_scores, on=['id_student', 'code_module', 'code_presentation'], how='left')
master_df = master_df.merge(vle_features, on=['id_student', 'code_module', 'code_presentation'], how='left')
master_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,avg_grade,total_clicks
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159,?,82.0,934.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,-53,?,66.4,1435.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,-92,12,NaN,281.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,-52,?,76.0,2158.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,-176,?,54.4,1034.0


In [16]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  object 
 1   code_presentation     32593 non-null  object 
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  object 
 4   region                32593 non-null  object 
 5   highest_education     32593 non-null  object 
 6   imd_band              32593 non-null  object 
 7   age_band              32593 non-null  object 
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  object 
 11  final_result          32593 non-null  object 
 12  date_registration     32593 non-null  object 
 13  date_unregistration   32593 non-null  object 
 14  avg_grade             25820 non-null  float64
 15  total_clicks       

In [17]:
#Quote from paper: " Following the data collection phase, we processed the data using an imputer and scalar. The imputer
# replaced all missing values (NA) with the median value of that specific feature"
#Import necessary capabilities
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [18]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  object 
 1   code_presentation     32593 non-null  object 
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  object 
 4   region                32593 non-null  object 
 5   highest_education     32593 non-null  object 
 6   imd_band              32593 non-null  object 
 7   age_band              32593 non-null  object 
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  object 
 11  final_result          32593 non-null  object 
 12  date_registration     32593 non-null  object 
 13  date_unregistration   32593 non-null  object 
 14  avg_grade             25820 non-null  float64
 15  total_clicks       

In [19]:
#Check any null values
print(master_df.isna().sum()) #Print how many null values by attribute

code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                   0
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
date_registration          0
date_unregistration        0
avg_grade               6773
total_clicks            3365
dtype: int64


In [20]:
#Convert date_registration and date_unregistration to int (data type as given by NAMES file)
#Example code from earlier: final_assessment['score'] = pd.to_numeric(final_assessment['score'], errors='coerce')

master_df['date_registration'] = pd.to_numeric(master_df['date_registration'], errors='coerce')
master_df['date_unregistration'] = pd.to_numeric(master_df['date_unregistration'], errors='coerce')

master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  object 
 1   code_presentation     32593 non-null  object 
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  object 
 4   region                32593 non-null  object 
 5   highest_education     32593 non-null  object 
 6   imd_band              32593 non-null  object 
 7   age_band              32593 non-null  object 
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  object 
 11  final_result          32593 non-null  object 
 12  date_registration     32548 non-null  float64
 13  date_unregistration   10072 non-null  float64
 14  avg_grade             25820 non-null  float64
 15  total_clicks       

In [21]:
#Pull out numerical columns that need to be adjusted
num_cols = ['date_registration', 'date_unregistration', 'avg_grade', 'total_clicks', 'num_of_prev_attempts', 'studied_credits']
median_imp = SimpleImputer(strategy='median')
master_df[num_cols] = median_imp.fit_transform(master_df[num_cols])
master_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,avg_grade,total_clicks
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0.0,240.0,N,Pass,-159.0,27.0,82.0,934.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0.0,60.0,N,Pass,-53.0,27.0,66.4,1435.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0.0,60.0,Y,Withdrawn,-92.0,12.0,76.0,281.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0.0,60.0,N,Pass,-52.0,27.0,76.0,2158.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0.0,60.0,N,Pass,-176.0,27.0,54.4,1034.0


In [22]:
print(type(num_cols))

<class 'list'>


In [23]:
# Quote from paper: "The scalar normalized the features within the range of 0 to 1, preventing large ranges from
# dominating the features and ensuring that the features were unit dependent."
#Import necessary capabilities
from sklearn.preprocessing import MinMaxScaler

In [24]:
mmscaler = MinMaxScaler(feature_range=(0,1))
master_df[num_cols] = mmscaler.fit_transform(master_df[num_cols])
master_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,avg_grade,total_clicks
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0.0,0.336,N,Pass,0.333333,0.484549,0.820,0.038653
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0.0,0.048,N,Pass,0.550102,0.484549,0.664,0.059408
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0.0,0.048,Y,Withdrawn,0.470348,0.466007,0.760,0.011600
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0.0,0.048,N,Pass,0.552147,0.484549,0.760,0.089361
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0.0,0.048,N,Pass,0.298569,0.484549,0.544,0.042796


In [25]:
# Quote from paper: "...the transformation of region, code module, and code presentation into columnar data was
# performed using one-hot encoding for these categories."
from sklearn.preprocessing import OneHotEncoder

In [26]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  object 
 1   code_presentation     32593 non-null  object 
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  object 
 4   region                32593 non-null  object 
 5   highest_education     32593 non-null  object 
 6   imd_band              32593 non-null  object 
 7   age_band              32593 non-null  object 
 8   num_of_prev_attempts  32593 non-null  float64
 9   studied_credits       32593 non-null  float64
 10  disability            32593 non-null  object 
 11  final_result          32593 non-null  object 
 12  date_registration     32593 non-null  float64
 13  date_unregistration   32593 non-null  float64
 14  avg_grade             32593 non-null  float64
 15  total_clicks       

In [27]:
cat_cols = ['gender', 'region', 'code_module', 'code_presentation', 'highest_education', 'imd_band', 'age_band', 'disability', ]
ohe = OneHotEncoder(sparse_output=False)
cat_data = master_df[cat_cols]
encoded_data = ohe.fit_transform(cat_data)
feature_names = ohe.get_feature_names_out(cat_cols)
encoded_df = pd.DataFrame(encoded_data, columns=feature_names)
master_df_num = master_df.drop(columns = cat_cols, axis = 1)
master_df_f = pd.concat([master_df_num, encoded_df], axis = 1)
master_df_f.head()

,id_student,num_of_prev_attempts,studied_credits,final_result,date_registration,date_unregistration,avg_grade,total_clicks,gender_F,gender_M,...,imd_band_60-70%,imd_band_70-80%,imd_band_80-90%,imd_band_90-100%,imd_band_?,age_band_0-35,age_band_35-55,age_band_55<=,disability_N,disability_Y
0,11391,0.0,0.336,Pass,0.333333,0.484549,0.820,0.038653,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1,28400,0.0,0.048,Pass,0.550102,0.484549,0.664,0.059408,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,30268,0.0,0.048,Withdrawn,0.470348,0.466007,0.760,0.011600,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,31604,0.0,0.048,Pass,0.552147,0.484549,0.760,0.089361,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,32885,0.0,0.048,Pass,0.298569,0.484549,0.544,0.042796,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [28]:
master_df_f.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 55 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id_student                                     32593 non-null  int64  
 1   num_of_prev_attempts                           32593 non-null  float64
 2   studied_credits                                32593 non-null  float64
 3   final_result                                   32593 non-null  object 
 4   date_registration                              32593 non-null  float64
 5   date_unregistration                            32593 non-null  float64
 6   avg_grade                                      32593 non-null  float64
 7   total_clicks                                   32593 non-null  float64
 8   gender_F                                       32593 non-null  float64
 9   gender_M                                       325

2. Build two prediction tasks:


*   Four-class classification: Fail / Pass / Withdrawn / Distinction
*   Two-class classification: Pass vs. Fail (merged)


In [29]:
#In this study, logistic regression was employed to classify data into two scenarios. The first scenario involved
#classifying the data into two categories: fail and pass . The second scenario involved
# classifying the data into four categories: fail , pass , withdrawn , and distinction
master_df_4C = master_df_f.copy()

four_class_map = {
    'Withdrawn': 1,
    'Distinction': 2,
    'Pass': 3,
    'Fail': 4
}

master_df_4C['results_4C'] = master_df_f['final_result'].map(four_class_map)
master_df_4C = master_df_4C.drop('final_result', axis = 1)

In [30]:
print(master_df_f['final_result'].unique())

['Pass' 'Withdrawn' 'Fail' 'Distinction']


In [31]:
class_map = {
    'Withdrawn': 1,
    'Distinction': 2,
    'Pass': 2,
    'Fail': 1
}

master_df_2C = master_df_f.copy()
master_df_2C['results_2C'] = master_df_f['final_result'].map(class_map)
master_df_2C = master_df_2C.drop('final_result', axis = 1)
master_df_2C.head()

,id_student,num_of_prev_attempts,studied_credits,date_registration,date_unregistration,avg_grade,total_clicks,gender_F,gender_M,region_East Anglian Region,...,imd_band_70-80%,imd_band_80-90%,imd_band_90-100%,imd_band_?,age_band_0-35,age_band_35-55,age_band_55<=,disability_N,disability_Y,results_2C
0,11391,0.0,0.336,0.333333,0.484549,0.820,0.038653,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,2
1,28400,0.0,0.048,0.550102,0.484549,0.664,0.059408,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2
2,30268,0.0,0.048,0.470348,0.466007,0.760,0.011600,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1
3,31604,0.0,0.048,0.552147,0.484549,0.760,0.089361,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2
4,32885,0.0,0.048,0.298569,0.484549,0.544,0.042796,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2


In [32]:
master_df_2C.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 55 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id_student                                     32593 non-null  int64  
 1   num_of_prev_attempts                           32593 non-null  float64
 2   studied_credits                                32593 non-null  float64
 3   date_registration                              32593 non-null  float64
 4   date_unregistration                            32593 non-null  float64
 5   avg_grade                                      32593 non-null  float64
 6   total_clicks                                   32593 non-null  float64
 7   gender_F                                       32593 non-null  float64
 8   gender_M                                       32593 non-null  float64
 9   region_East Anglian Region                     325

In [33]:
master_df_4C.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 55 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id_student                                     32593 non-null  int64  
 1   num_of_prev_attempts                           32593 non-null  float64
 2   studied_credits                                32593 non-null  float64
 3   date_registration                              32593 non-null  float64
 4   date_unregistration                            32593 non-null  float64
 5   avg_grade                                      32593 non-null  float64
 6   total_clicks                                   32593 non-null  float64
 7   gender_F                                       32593 non-null  float64
 8   gender_M                                       32593 non-null  float64
 9   region_East Anglian Region                     325

3. Train the same models used in the study:

*   Logistic Regression
*   Random Forest Classifier


In [34]:
# Quote from paper: "The model selection stage was focused on, starting with the division of the data into train
# and test sets using a 75/25 split."
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc

In [35]:
#Logistic Regression - 4C
X_four = master_df_4C.drop('results_4C', axis = 1)
y_four = master_df_4C['results_4C']

X_train_l4, X_test_l4, y_train_l4, y_test_l4 = train_test_split(X_four, y_four, test_size=0.25, random_state=42)
log_mod_4C = LogisticRegression(solver = 'saga', max_iter = 100)


In [36]:
log_mod_4C.fit(X_train_l4, y_train_l4)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(solver='saga')

In [37]:
y_pred_l4 = log_mod_4C.predict(X_test_l4)

In [38]:
#Two class logistic regression

X_two = master_df_2C.drop('results_2C', axis = 1)
y_two = master_df_2C['results_2C']

X_train_l2, X_test_l2, y_train_l2, y_test_l2 = train_test_split(X_two, y_two, test_size=0.25, random_state=42)

log_mod_2C = LogisticRegression(solver = 'saga', max_iter = 100)

In [39]:
log_mod_2C.fit(X_train_l2, y_train_l2)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(solver='saga')

In [40]:
y_pred_l2 = log_mod_2C.predict(X_test_l2)

In [41]:
#Four Class Random Forest
rf_4C = RandomForestClassifier(n_estimators = 100, random_state = 42)
X_train_r4, X_test_r4, y_train_r4, y_test_r4 = train_test_split(X_four, y_four, test_size=0.25, random_state=42)
rf_4C.fit(X_train_r4, y_train_r4)
y_pred_rf4 = rf_4C.predict(X_test_r4)

In [42]:
#Two Class Random Forest
rf_2C = RandomForestClassifier(n_estimators = 100, random_state = 42)
X_train_r2, X_test_r2, y_train_r2, y_test_r2 = train_test_split(X_two, y_two, test_size=0.25, random_state=42)
rf_2C.fit(X_train_r2, y_train_r2)
y_pred_rf2 = rf_2C.predict(X_test_r2)

4. Evaluate the models:

*   Accuracy
*   F1 Score


*   Confusion matrix

In [43]:
#Evaluation function
def eval(model, y_test, y_pred):
  acc = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='weighted')
  cm = confusion_matrix(y_test, y_pred)

  print("Accuracy:", acc)
  print("F1 Score:", f1)
  print("Confusion Matrix:")
  print(cm)

In [44]:
print("Four Class - Logistic Regression")
eval(log_mod_4C, y_test_l4, y_pred_l4)

Four Class - Logistic Regression
Accuracy: 0.3773469137317462
F1 Score: 0.2067608267507341
Confusion Matrix:
[[   0    0 2450    0]
 [   0    0  775    0]
 [   0    0 3075    0]
 [   0    0 1849    0]]


In [45]:
print("Two Class - Logistic Regression")
eval(log_mod_2C, y_test_l2, y_pred_l2)

Two Class - Logistic Regression
Accuracy: 0.5275493925635047
F1 Score: 0.36438541751775494
Confusion Matrix:
[[4299    0]
 [3850    0]]


In [46]:
print("Four Class - Random Forest")
eval(rf_4C, y_test_r4, y_pred_rf4)

Four Class - Random Forest
Accuracy: 0.8089336114860719
F1 Score: 0.802628307333603
Confusion Matrix:
[[2396    0    4   50]
 [   0  320  446    9]
 [   0  134 2694  247]
 [  13   13  641 1182]]


In [47]:
print("Two Class - Random Forest")
eval(rf_2C, y_test_r2, y_pred_rf2)

Two Class - Random Forest
Accuracy: 0.8890661430850411
F1 Score: 0.8891497561404174
Confusion Matrix:
[[3725  574]
 [ 330 3520]]


5. Compare the models:

*   Which model performed better?
*   Why?


*   How does performance differ between the two-class and four-class tasks?


ANALYSIS PERFORMED IN PAPER.